In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

FILE_PATH = "../data/final_glacier_ml_dataset.parquet"
df = pd.read_parquet(FILE_PATH)


In [ ]:
print("--- Dataset Overview ---")
print(f"Total Rows (Individual Observations): {len(df)}")
print(f"Columns Available: {df.columns.tolist()}")

In [ ]:
print("\n--- Missing Value Check ---")
print(df[['area_m2', 'sla', 'B2', 'NDSI', 'elev_mean']].isnull().sum())

In [ ]:
obs_per_satellite = df["satellite"].value_counts().sort_index()

print(obs_per_satellite)

In [ ]:
order = ["sentinel2", "landsat5", "landsat8"]
obs_per_satellite = df["satellite"].value_counts().reindex(order, fill_value=0)

obs_per_satellite.plot(kind="bar")
plt.ylabel("Number of observations")
plt.xlabel("Satellite")
plt.title("Observations per satellite")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
images_per_year = df.groupby(df["observation_start"].dt.year).size()

print(images_per_year)

images_per_year.plot(kind="bar")
plt.ylabel("Number of images / observations")
plt.xlabel("Year")
plt.title("Images over time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
obs_per_glacier = df.groupby("sgi-id").size().sort_values(ascending=False)

print(df["sgi-id"].unique())
print(obs_per_glacier)

In [ ]:
obs_per_glacier_sat = (
    df.groupby(["sgi-id", "satellite"])
      .size()
      .unstack(fill_value=0)
)

print(obs_per_glacier_sat)

obs_per_glacier_sat.plot(kind="bar", stacked=True, figsize=(12,5))
plt.ylabel("Number of observations")
plt.xlabel("Glacier")
plt.title("Observations per glacier by satellite")
plt.tight_layout()
plt.show()

In [ ]:
print("Total observations:", len(df))
print("\nObservations per satellite:")
print(df["satellite"].value_counts())

print("\nObservations per glacier:")
print(df["sgi-id"].value_counts())

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())

In [ ]:
obs_per_glacier.plot(kind="bar", figsize=(12,4))
plt.ylabel("Number of observations")
plt.xlabel("Glacier")
plt.title("Observations per glacier")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='sla', y='area_m2', hue='satellite', alpha=0.7)
plt.title(" SLA vs Snow Area")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
df.head(50)

In [ ]:
import geemap
import pandas as pd
import ee

rasters = pd.read_pickle("../data/raster_verification.pkl")
df_rasters = pd.DataFrame(rasters)

def inspect_observation(obs_id):
    
    row = df_rasters[df_rasters['obs_id'] == obs_id]
    
    if row.empty:
        print(f"Observation {obs_id} not found in the pickle file.")
        return

    
    mask_img = row.iloc[0]['mask_image']
    
    print(f"Inspecting Observation: {obs_id}")
    
    Map = geemap.Map()
    Map.centerObject(mask_img, 14)
    
    Map.add_basemap('SATELLITE')

    mask_vis = {'min': 0, 'max': 1, 'palette': ['black', 'white'], 'opacity': 0.6}
    Map.addLayer(mask_img, mask_vis, 'Snow Mask (Median)')
    
    return Map


inspect_observation("A10g-05_1985-10-01_1986-09-30")

In [ ]:
import ee
import geemap
import pandas as pd

ee.Initialize(project="project-8e6c1255-803c-4395-88f")

df_rasters = pd.read_pickle("../data/raster_verification.pkl")

def show_mask_from_pickle(obs_id, lon=8.4, lat=46.8, zoom=11):
    row = df_rasters[df_rasters["obs_id"] == obs_id]
    if row.empty:
        print(f"{obs_id} not found.")
        return

    mask_img = row.iloc[0]["mask_image"]

    m = geemap.Map()
    m.setCenter(lon, lat, zoom)  

    m.add_basemap("SATELLITE")

    try:
        m.addLayer(
            ee.Image(mask_img).selfMask(),
            {"palette": ["white"], "opacity": 0.7},
            f"Mask {obs_id}"
        )
        return m
    except Exception as e:
        print(f"Could not display mask image: {e}")
        print("The pickled ee.Image is not reusable enough for map display.")

In [ ]:
df_rasters.columns

In [ ]:
import pickle
with open("../data/raster_verification.pkl", "rb") as f:
    data = pickle.load(f)

print(data.dtypes)